# Binary LinearSVC Training With Local MLflow



In [1]:
import os
import gc
import sys
import time
import pickle
import warnings
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
import mlflow
from tqdm import tqdm
from scipy.sparse import hstack, csr_matrix
from sklearn.svm import LinearSVC
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import precision_score, recall_score, f1_score, classification_report

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'src').exists() and (PROJECT_ROOT.parent / 'src').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
DEFAULT_MODEL_CONFIG = PROJECT_ROOT / 'src' / 'config_s3.json'
if 'MODEL_CONFIG' not in os.environ and DEFAULT_MODEL_CONFIG.exists():
    os.environ['MODEL_CONFIG'] = str(DEFAULT_MODEL_CONFIG)
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

from services.utils import Boto3Reader, load_config
from services.text_preprocessor import LinearSVMPreprocessorSI

warnings.simplefilter(action='ignore')

f:\HSE\year-project\toxic-messages-data-platform\toxic-messages-handling-project\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# setup constants and load data: 
USE_S3 = False
USE_MLFLOW = True
host = "http://localhost:5000"

DATA_PATH = 'dump_features_include_numeric_1.csv'
CONFIG_PATH = os.environ.get('MODEL_CONFIG', 'src/config_s3.json')
TEXT_COL = 'text_raw'
TARGET_COL = 'is_toxic'
RANDOM_STATE = 42
N_ROWS = 400000

if USE_S3:
    reader = Boto3Reader(CONFIG_PATH)
    df = reader.get_boto3_csv(object_path=DATA_PATH)
    del reader
else:
    df = pd.read_csv(DATA_PATH, index_col=0)

df = df.dropna(subset=[TEXT_COL, TARGET_COL]).copy()
df[TARGET_COL] = df[TARGET_COL].astype(int)

if N_ROWS is not None and len(df) > N_ROWS:
    cls_counts = df[TARGET_COL].value_counts()
    sample_weights = df[TARGET_COL].map(lambda x: 1.0 / cls_counts[x])
    df = df.sample(n=N_ROWS, weights=sample_weights, random_state=RANDOM_STATE)

print('dataset shape:', df.shape)
print('class balance:')
print(df[TARGET_COL].value_counts(normalize=True).sort_index())

dataset shape: (330914, 35)
class balance:
is_toxic
0    0.783974
1    0.216026
Name: proportion, dtype: float64


In [3]:
# init inference-equivalent preprocessor: 
full_config = load_config(CONFIG_PATH)
predictor_config = full_config['predictors'][0]
preprocessor = LinearSVMPreprocessorSI(config=predictor_config)

def preprocess_texts(texts, preproc):
    processed_texts = []
    processed_num_features = []
    for text in tqdm(texts, total=len(texts)):
        text_preprocessed, num_features = preproc.preprocess(str(text))
        processed_texts.append(text_preprocessed)
        processed_num_features.append(num_features)
    return processed_texts, processed_num_features

enc_emoj_file models\v0\encoding_emoji.json


In [4]:
# split data and build train/test features: 
train_df, test_df = train_test_split(
    df,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=df[TARGET_COL]
)

train_texts, train_num_parts = preprocess_texts(train_df[TEXT_COL].tolist(), preprocessor)
test_texts, test_num_parts = preprocess_texts(test_df[TEXT_COL].tolist(), preprocessor)

vectorizer = TfidfVectorizer(
    ngram_range=(1, 2),
    min_df=3,
    max_df=0.9
)

X_train_text = vectorizer.fit_transform(train_texts)
X_test_text = vectorizer.transform(test_texts)

X_train_num = csr_matrix(np.vstack([x.toarray() for x in train_num_parts]))
X_test_num = csr_matrix(np.vstack([x.toarray() for x in test_num_parts]))

X_train = hstack([X_train_text, X_train_num]).tocsr()
X_test = hstack([X_test_text, X_test_num]).tocsr()

y_train = train_df[TARGET_COL].values
y_test = test_df[TARGET_COL].values

print('X_train shape:', X_train.shape)
print('X_test shape:', X_test.shape)

100%|██████████| 66183/66183 [02:58<00:00, 369.75it/s]


X_train shape: (264731, 259217)
X_test shape: (66183, 259217)


In [5]:
# !mlflow ui

In [8]:
EXPERIMENT_NAME = 'linear-svc-binary-inference-eq'
SAVE_DIR = 'exp_linear_svc'

os.makedirs(SAVE_DIR, exist_ok=True)
mlflow.set_tracking_uri(host)
mlflow.set_experiment(EXPERIMENT_NAME)

timestamp = datetime.now().strftime('%y-%d-%H_%M_%S')
run_save_dir = os.path.join(SAVE_DIR, f'run_{timestamp}')
os.makedirs(run_save_dir, exist_ok=True)

model = LinearSVC(class_weight='balanced', random_state=RANDOM_STATE)

try:
    if USE_MLFLOW:
        mlflow.start_run()
        mlflow.set_tag('tag', 'PRD')
        mlflow.log_params({
            'model_name': 'LinearSVC',
            'class_weight': 'balanced',
            'random_state': RANDOM_STATE,
            'test_size': 0.2,
            'text_column': TEXT_COL,
            'target_column': TARGET_COL,
            'vectorizer_ngram_range': '(1,2)',
            'vectorizer_min_df': 3,
            'vectorizer_max_df': 0.9,
            'preprocessor_class': 'LinearSVMPreprocessorSI'
        })

    start_time = time.time()
    model.fit(X_train, y_train)
    train_time = time.time() - start_time

    y_pred = model.predict(X_test)
    metrics = {
        'test_precision_weighted': precision_score(y_test, y_pred, average='weighted', zero_division=0),
        'test_recall_weighted': recall_score(y_test, y_pred, average='weighted', zero_division=0),
        'test_f1_weighted': f1_score(y_test, y_pred, average='weighted', zero_division=0),
        'test_f1_macro': f1_score(y_test, y_pred, average='macro', zero_division=0),
        'train_time_sec': train_time
    }

    print(classification_report(y_test, y_pred))
    print('metrics:', {k: round(v, 4) for k, v in metrics.items()})

    model_path = os.path.join(run_save_dir, 'linear_svc_model.pkl')
    vectorizer_path = os.path.join(run_save_dir, 'linear_svc_vectorizer.pkl')
    with open(model_path, 'wb') as f:
        pickle.dump(model, f)
    with open(vectorizer_path, 'wb') as f:
        pickle.dump(vectorizer, f)

    if USE_MLFLOW:
        mlflow.log_metrics(metrics)
        mlflow.log_artifact(model_path, artifact_path='model_weights')
        mlflow.log_artifact(vectorizer_path, artifact_path='encoder')

    print('saved artifacts:', run_save_dir)

except Exception as ex:
    print(ex)
finally:
    if USE_MLFLOW:
        mlflow.end_run()
    gc.collect()

              precision    recall  f1-score   support

           0       0.96      0.97      0.96     51886
           1       0.87      0.86      0.87     14297

    accuracy                           0.94     66183
   macro avg       0.92      0.91      0.91     66183
weighted avg       0.94      0.94      0.94     66183

metrics: {'test_precision_weighted': 0.9419, 'test_recall_weighted': 0.9422, 'test_f1_weighted': 0.942, 'test_f1_macro': 0.9141, 'train_time_sec': 18.7452}
saved artifacts: exp_linear_svc\run_26-30-22_14_38
🏃 View run colorful-worm-288 at: http://localhost:5000/#/experiments/9/runs/63cc099790374a6ea327f93a45c953d0
🧪 View experiment at: http://localhost:5000/#/experiments/9
